# Figure 4, Panel D

Barplots showing expression (log2(TPM+1)) of canonical IFNg- and Dex-induced genes at 1h and 6h with stimulus alone or after 0.5h pretreatment with BAFinh (10µM). Expression values for individual replicates indicated by points. P-values comparing Stimulus expression to BAFinh -> Stimulus are from one-sided Student's t-tests.

*Outputs: paneld_ifny.pdf, paneld_dex.pdf*

# Setup

In [1]:
# load packages
library(tidyverse)
library(RColorBrewer)
library(patchwork)
library(here)
here::i_am("figures/Figure4/Panel_D/PanelD_example_genes.ipynb")

theme_set(theme_bw() + theme(panel.grid=element_blank(), text=element_text(size=15)))
sessionInfo()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/figures/Figure4/Panel_D

here() starts at /mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics



R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] here_1.0.2         patchwork_1.3.0    RColorBrewer_1.1-3 lubridate_1.9.4   
 [5] forcats_1.0.0      stringr_1.5.1      dplyr_1.1.4        purrr_1.0.2       
 [9] readr_2.1.5        tidyr_1.3.1        tibble_3.2.1       ggplot2_3.5.2     
[13] tidyverse_2.0.0   

loaded via a namespace (an

# Plotting function

# Read data

In [2]:
# TPMs
tpms <- read.delim(
  here(
    "figures", "Figure4",
    "IFNy_Dex_DEGs_TPMs.tsv"
  ),
  sep = "\t"
)

# gene-level metadata
row.data <- read.delim(
  here(
    "figures", "Figure4",
    "IFNy_Dex_DEGs_rowdata.tsv"
  ),
  sep = "\t"
)

# sample-level metadata
col.data <- read.delim(
  here(
    "figures", "Figure4",
    "IFNy_Dex_DEGs_coldata.tsv"
  ),
  sep = "\t"
)

colnames(tpms) <- paste0(col.data$group, "_", col.data$replicate)

# Generate plots

In [7]:
plot_tpms <- function(tpms, rowdata, gene, conditions){
    temp <- tpms %>% data.frame() %>% mutate(gene_name=rowdata$gene_name) #add gene name from row metadata
    temp <- temp %>% filter(gene_name == gene) #filter to only include gene of interest
    temp <- temp %>% 
                pivot_longer(!gene_name) %>% #make into tidy format
                mutate(condition = gsub("_r.*$", "", name), #extract condition (dose + time) as well as dose and time separately
                    time=gsub("^.*_", "", condition), 
                    treatment=gsub("_T.*$", "", condition)) %>% 
                filter(treatment %in% conditions) 
    temp <- temp %>% mutate(condition=case_when(grepl("Veh", condition) ~ "Veh", TRUE ~ condition), #combine vehicle treated samples into one category
                            time=case_when(grepl("Veh", condition) ~ "T0", TRUE ~ time),
                           value=log2(value+1)) #log2 transform TPMs for plotting

    #statistical tests: is induction at 1h or 6h reduced with BAFinh?
    p_t1 <- t.test(temp[temp$condition=="BRM_IFN_T1",]$value, temp[temp$condition=="IFN_T1",]$value, alternative="less")$p.value 
    p_t6 <- t.test(temp[temp$condition=="BRM_IFN_T6",]$value, temp[temp$condition=="IFN_T6",]$value, alternative="less")$p.value 

    #plot results
    plot <- ggplot(temp, aes(x=interaction(condition,time),y=value, fill=condition)) + 
        stat_summary(fun=mean, geom="col", width=.75, color="black", position=position_dodge()) +  #mean TPM as column
        geom_point(position=position_jitter(height=0,width=.1),  size=2, color="grey") + #individual replicates as points
        ggtitle(gene, subtitle=paste0("p(1h) = ", format(p_t1, digits=3), "\np(6h) = ", format(p_t6, digits=3))) + #report gene name and p-values in title
        ylab("Log2(TPM+1)") + coord_cartesian(ylim=c(0,9)) + scale_y_continuous(breaks=c(0,4,8))
    return(list(temp, plot))
}

IFNy_targets <- c("GBP5", "IRF8", "CXCL10","SOCS1")
#plot_tpms(tpms, row.data, IFNy_targets[1], c("Veh", "IFN"))
p1 <- plot_tpms(tpms, row.data, IFNy_targets[1], c("Veh", "IFN", "BRM_IFN")) #+ coord_cartesian(ylim=c(0,7))
p2 <- plot_tpms(tpms, row.data, IFNy_targets[2], c("Veh", "IFN", "BRM_IFN")) #+ coord_cartesian(ylim=c(0,7))
p3 <- plot_tpms(tpms, row.data, IFNy_targets[3], c("Veh", "IFN", "BRM_IFN")) #+ coord_cartesian(ylim=c(0,7))
p4 <- plot_tpms(tpms, row.data, IFNy_targets[4], c("Veh", "IFN", "BRM_IFN")) #+ coord_cartesian(ylim=c(0,7))

(p1[[2]] | p3[[2]] | p4[[2]] | p2[[2]]) + plot_layout(guides="collect")
ggsave("paneld_ifny.pdf",width=10,height=4)

In [8]:
plot_tpms <- function(tpms, rowdata, gene, conditions){
    temp <- tpms %>% data.frame() %>% mutate(gene_name=rowdata$gene_name) #add gene name from row metadata
    temp <- temp %>% filter(gene_name == gene) #filter to only include gene of interest
    temp <- temp %>% 
                pivot_longer(!gene_name) %>% #make into tidy format
                mutate(condition = gsub("_r.*$", "", name), #extract condition (dose + time) as well as dose and time separately
                    time=gsub("^.*_", "", condition), 
                    treatment=gsub("_T.*$", "", condition)) %>% 
                filter(treatment %in% conditions) 
    temp <- temp %>% mutate(condition=case_when(grepl("Veh", condition) ~ "Veh", TRUE ~ condition), #combine vehicle treated samples into one category
                            time=case_when(grepl("Veh", condition) ~ "T0", TRUE ~ time),
                           value=log2(value+1)) #log2 transform TPMs for plotting

    #statistical tests: is induction at 1h or 6h reduced with BAFinh?
    p_t1 <- t.test(temp[temp$condition=="BRM_Dex_T1",]$value, temp[temp$condition=="Dex_T1",]$value, alternative="less")$p.value 
    p_t6 <- t.test(temp[temp$condition=="BRM_Dex_T6",]$value, temp[temp$condition=="Dex_T6",]$value, alternative="less")$p.value 

    #plot results
    plot <- ggplot(temp, aes(x=interaction(condition,time),y=value, fill=condition)) + 
        stat_summary(fun=mean, geom="col", width=.75, color="black", position=position_dodge()) +  #mean TPM as column
        geom_point(position=position_jitter(height=0,width=.1),  size=2, color="grey") + #individual replicates as points
        ggtitle(gene, subtitle=paste0("p(1h) = ", format(p_t1, digits=3), "\np(6h) = ", format(p_t6, digits=3))) + #report gene name and p-values in title
        ylab("Log2(TPM+1)") + coord_cartesian(ylim=c(0,9)) + scale_y_continuous(breaks=c(0,4,8))
    return(list(temp, plot))
}

Dex_targets <- c("DDIT4", "PER1", "TSC22D3","FKBP5")
#plot_tpms(tpms, row.data, IFNy_targets[1], c("Veh", "IFN"))
p1 <- plot_tpms(tpms, row.data, Dex_targets[1], c("Veh", "Dex", "BRM_Dex")) #+ coord_cartesian(ylim=c(0,7))
p2 <- plot_tpms(tpms, row.data, Dex_targets[2], c("Veh", "Dex", "BRM_Dex")) #+ coord_cartesian(ylim=c(0,7))
p3 <- plot_tpms(tpms, row.data, Dex_targets[3], c("Veh", "Dex", "BRM_Dex")) #+ coord_cartesian(ylim=c(0,7))
p4 <- plot_tpms(tpms, row.data, Dex_targets[4], c("Veh", "Dex", "BRM_Dex")) #+ coord_cartesian(ylim=c(0,7))

(p2[[2]] | p3[[2]] | p1[[2]] | p4[[2]]) + plot_layout(guides="collect")
ggsave("panelD_dex.pdf",width=10,height=4)